# Train BanglaBERT to answer Physics questions

This notebook fine-tunes `csebuetnlp/banglabert` for **extractive question answering**. Given a Bengali question and a retrieved textbook passage, the model learns the start and end tokens of the answer.

Choose **Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
# Colab/GPU guard: training is intentionally disabled on the local computer.
try:
    import google.colab
except ImportError as exc:
    raise RuntimeError('Run this notebook in Google Colab, not locally.') from exc
import torch
assert torch.cuda.is_available(), 'Enable T4 GPU under Runtime > Change runtime type'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!pip -q install 'transformers==4.46.3' 'datasets==3.2.0' 'accelerate==1.2.1' 'sentencepiece>=0.2.0'

## Upload the answer dataset

Upload `dataset/physics_qa_234.json` when prompted.

In [ ]:
import json
from collections import Counter, defaultdict
from pathlib import Path
from google.colab import files

uploaded = files.upload()
dataset_name = next((name for name in uploaded if name == 'physics_qa_234.json'), None)
assert dataset_name, 'Upload physics_qa_234.json'
document = json.loads(Path(dataset_name).read_text(encoding='utf-8'))
examples = document['examples']
assert len(examples) == 234, f'Expected 234 examples, found {len(examples)}'
assert Counter(row['chapter_id'] for row in examples) == {chapter: 18 for chapter in range(1, 14)}
for row in examples:
    answer = row['answers']['text'][0]
    start = row['answers']['answer_start'][0]
    assert row['context'][start:start + len(answer)] == answer, row['id']
print('Validated 234 answer spans across 13 chapters')

In [ ]:
# Deterministic chapter-balanced split: 15 train + 3 validation examples per chapter.
from datasets import Dataset, DatasetDict
by_chapter = defaultdict(list)
for row in examples:
    by_chapter[row['chapter_id']].append(row)
train_rows, validation_rows = [], []
for chapter in range(1, 14):
    rows = sorted(by_chapter[chapter], key=lambda row: row['id'])
    train_rows.extend(rows[:15])
    validation_rows.extend(rows[15:])
raw = DatasetDict({
    'train': Dataset.from_list(train_rows),
    'validation': Dataset.from_list(validation_rows),
})
print(raw)

In [ ]:
from transformers import AutoTokenizer
MODEL_CHECKPOINT = 'csebuetnlp/banglabert'
MAX_LENGTH = 384
DOC_STRIDE = 96
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, use_fast=True)
assert tokenizer.is_fast, 'Answer offsets require a fast tokenizer'

In [ ]:
def prepare_features(batch):
    tokenized = tokenizer(
        [question.strip() for question in batch['question']],
        batch['context'],
        truncation='only_second',
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding='max_length',
    )
    sample_map = tokenized.pop('overflow_to_sample_mapping')
    offsets = tokenized.pop('offset_mapping')
    starts, ends = [], []
    for feature_index, offset in enumerate(offsets):
        sample_index = sample_map[feature_index]
        answer = batch['answers'][sample_index]
        answer_start = answer['answer_start'][0]
        answer_end = answer_start + len(answer['text'][0])
        sequence_ids = tokenized.sequence_ids(feature_index)
        cls_index = tokenized['input_ids'][feature_index].index(tokenizer.cls_token_id)
        context_start = 0
        while sequence_ids[context_start] != 1:
            context_start += 1
        context_end = len(sequence_ids) - 1
        while sequence_ids[context_end] != 1:
            context_end -= 1
        if offset[context_start][0] > answer_start or offset[context_end][1] < answer_end:
            starts.append(cls_index)
            ends.append(cls_index)
            continue
        while context_start < len(offset) and offset[context_start][1] <= answer_start:
            context_start += 1
        while context_end >= 0 and offset[context_end][0] >= answer_end:
            context_end -= 1
        starts.append(context_start)
        ends.append(context_end)
    tokenized['start_positions'] = starts
    tokenized['end_positions'] = ends
    return tokenized

columns = raw['train'].column_names
tokenized = raw.map(prepare_features, batched=True, remove_columns=columns)
print(tokenized)

In [ ]:
from google.colab import drive
from transformers import (
    AutoModelForQuestionAnswering, DataCollatorWithPadding,
    EarlyStoppingCallback, Trainer, TrainingArguments,
)
drive.mount('/content/drive')
OUTPUT_DIR = '/content/drive/MyDrive/pathshongi-banglabert-physics-qa'
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_CHECKPOINT)
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR, learning_rate=2e-5,
    per_device_train_batch_size=8, per_device_eval_batch_size=16,
    gradient_accumulation_steps=2, num_train_epochs=10,
    weight_decay=0.01, warmup_ratio=0.1,
    evaluation_strategy='epoch', save_strategy='epoch',
    load_best_model_at_end=True, metric_for_best_model='eval_loss',
    greater_is_better=False, fp16=True, report_to='none',
    seed=42, save_total_limit=2, logging_steps=10,
)
trainer = Trainer(
    model=model, args=training_args,
    train_dataset=tokenized['train'], eval_dataset=tokenized['validation'],
    tokenizer=tokenizer, data_collator=DataCollatorWithPadding(tokenizer),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [ ]:
result = trainer.train()
print(result.metrics)
print(trainer.evaluate())
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
Path(OUTPUT_DIR, 'reader_metadata.json').write_text(
    json.dumps({'task': 'extractive-question-answering', 'max_length': MAX_LENGTH, 'doc_stride': DOC_STRIDE}, indent=2),
    encoding='utf-8',
)
print('Saved:', OUTPUT_DIR)

## Held-out answer check

This reports exact match and token F1 on the 39 validation examples. The dataset is small, so treat these as development metrics rather than a final benchmark.

In [ ]:
import re
from collections import Counter
from transformers import pipeline
reader = pipeline('question-answering', model=OUTPUT_DIR, tokenizer=OUTPUT_DIR, device=0)
def normalized(text):
    return ' '.join(re.findall(r'[\w\u0980-\u09FF]+', text.casefold()))
def token_f1(prediction, truth):
    predicted, expected = normalized(prediction).split(), normalized(truth).split()
    overlap = sum((Counter(predicted) & Counter(expected)).values())
    if not predicted or not expected:
        return float(predicted == expected)
    if overlap == 0:
        return 0.0
    precision, recall = overlap / len(predicted), overlap / len(expected)
    return 2 * precision * recall / (precision + recall)
predictions = []
for row in validation_rows:
    output = reader(question=row['question'], context=row['context'], max_answer_len=128)
    truth = row['answers']['text'][0]
    predictions.append((row, output['answer'], truth, output['score']))
exact = sum(normalized(pred) == normalized(truth) for _, pred, truth, _ in predictions) / len(predictions)
f1 = sum(token_f1(pred, truth) for _, pred, truth, _ in predictions) / len(predictions)
print({'exact_match': round(exact, 4), 'token_f1': round(f1, 4)})
for row, pred, truth, score in predictions[:5]:
    print('\nQ:', row['question'], '\nExpected:', truth, '\nPredicted:', pred, '\nConfidence:', round(score, 4))